# 02 Demand-gap score
Weighted score per LSOA (higher = bigger gap between need and provision), plus sensitivity analysis.
Input: `data/clean/lsoa_base.parquet`. Run from the `notebooks/` folder.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
CLEAN = Path("../data/clean")
df = pd.read_parquet(CLEAN / "lsoa_base.parquet")
freq = pd.read_parquet(CLEAN / "lsoa_frequency.parquet")   # from notebook 04 (GTFS)
df = df.merge(freq, on="LSOA21CD", how="left")
df.shape

## 1. Components
Each oriented so higher = worse provision or higher need.

`supply` is now **real service frequency**: mean departures per hour across the three time bands (from GTFS), per 1,000 residents, log-scaled. Zero service gives the worst value.

In [ ]:
df["freq_mean"] = df[["freq_morning", "freq_off_peak", "freq_evening"]].mean(axis=1)
df["freq_per_1000"] = df["freq_mean"] / df["population"] * 1000
df["stops_per_1000"] = df["n_stops"] / df["population"] * 1000
comp = pd.DataFrame({
    "access": df["dist_nearest_stop_m"],   # far from a stop
    "supply": -np.log1p(df["freq_per_1000"]),   # little service per person (GTFS)
    "density": df["pop_density"],          # many people to serve
    "deprivation": df["imd_score"],        # more deprived
}, index=df.index)
comp.describe().T

## 2. Normalise (0-1)
Winsorise at the 1st and 99th percentile, then min-max.

In [ ]:
def norm01(s, lo=0.01, hi=0.99):
    a, b = s.quantile([lo, hi])
    return (s.clip(a, b) - a) / (b - a)
N = comp.apply(norm01)
N.describe().T[["min","50%","max"]]

## 3. Weighted score
Baseline weights are **placeholders**: justify from the literature in the report.

In [ ]:
W = {"access": 0.30, "supply": 0.30, "density": 0.20, "deprivation": 0.20}
def score(weights):
    w = pd.Series(weights)[N.columns]
    return (N * (w / w.sum())).sum(axis=1)
df["gap_score"] = score(W)
df["gap_rank"] = df["gap_score"].rank(ascending=False, method="first").astype(int)
df["gap_top10"] = df["gap_rank"] <= int(0.10 * len(df))
df["gap_score"].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
df["gap_score"].hist(bins=60, ax=ax); ax.set_title("Demand-gap score (all England LSOAs)")
ax.set_xlabel("score (0-1)"); plt.tight_layout()

In [ ]:
df.groupby("Urban_rura")["gap_top10"].mean().round(3)

## 4. Sensitivity analysis
500 random weight sets (Dirichlet). Compare each ranking with the baseline: Spearman correlation and overlap of the top 10%.

In [ ]:
rng = np.random.default_rng(42)
base_top = set(df.index[df["gap_top10"]])
base_rank = df["gap_score"].rank()
rows = []
for i in range(500):
    w = dict(zip(N.columns, rng.dirichlet(np.ones(len(N.columns)))))
    s = score(w)
    top = set(s.nlargest(len(base_top)).index)
    rows.append({"run": i, **w,
                 "spearman_vs_base": s.rank().corr(base_rank),
                 "top10_overlap": len(top & base_top) / len(base_top)})
sens = pd.DataFrame(rows)
sens[["spearman_vs_base","top10_overlap"]].describe().round(3)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,3.5))
sens["spearman_vs_base"].hist(bins=30, ax=ax[0]); ax[0].set_title("Rank correlation with baseline")
sens["top10_overlap"].hist(bins=30, ax=ax[1]); ax[1].set_title("Top-10% overlap with baseline")
plt.tight_layout()

In [ ]:
# Named alternative weightings, compared with the baseline
alts = {
    "Equal weights (25/25/25/25)": {"access": .25, "supply": .25, "density": .25, "deprivation": .25},
    "Provision only (50/50/0/0)": {"access": .5, "supply": .5, "density": 0, "deprivation": 0},
    "Need-heavy (15/15/35/35)": {"access": .15, "supply": .15, "density": .35, "deprivation": .35},
}
rows = []
for name, w_ in alts.items():
    s_ = score(w_)
    top = set(s_.nlargest(len(base_top)).index)
    rows.append({"weights": name, "spearman_vs_baseline": s_.rank().corr(base_rank), "top10_overlap": len(top & base_top) / len(base_top)})
pd.DataFrame(rows).round(3)

## 5. Save

In [ ]:
df.to_parquet(CLEAN / "lsoa_score.parquet", index=False)
sens.to_csv(CLEAN / "sensitivity.csv", index=False)
print("saved")